# XGBoost Model Development and Evaluation Script

---

## Modeling Pipeline

- **Baseline Model Development:** A standard model was trained using default/reasonable hyperparameters to establish a performance benchmark.

- **Hyperparameter Optimization:** Each baseline model was optimized using two complementary techniques:
  - ***`Optuna`***: Efficient, adaptive search with pruning for rapid convergence to high-performing configurations.
  - ***`RandomizedSearchCV`***: Sklearn-compatible baseline optimization for fair comparison and reproducibility.

- **Cross-Validation and Robustness Assessment:** Each model variant was evaluated using ***`TimeSeriesSplit`*** to preserve temporal order and prevent look-ahead bias. Metrics were aggregated across folds to assess stability.

- **Overfitting Analysis:** A detailed comparison between cross-validation metrics and test set results was conducted. Additional metrics, including ***`RMSE ratio`*** and ***`R² gap`***, were computed to quantify overfitting and assess model generalization. ***`Directional accuracy`*** and ***`financial metrics`*** (Sharpe Ratio, Max Drawdown) were also calculated for trading-relevant evaluation.

---

## Persisted Artifacts

To ensure reproducibility, transparency, and extendability, the following artifacts have been saved for **each model**:

- **Optimized Model Performance:** Individual CSV files capturing the performance of each model variant:
    - ***XGBoost (Baseline)***
    - ***XGBoost (Optuna)***
    - ***XGBoost (RandomizedSearchCV)***

- **Best Variation Performance:** A CSV file containing only the metrics of the best-performing variation per model.

- **Summary of Model Performance:** A consolidated, extendable CSV file (`AllModel_OverallPerformance.csv`) including:
    - Cross-validation results (`CV MSE`, `CV MAE`, `CV RMSE`, `CV R²`, `CV MAPE`)
    - Test set results (`Test MSE`, `Test MAE`, `Test RMSE`, `Test R²`, `Test MAPE`)
    - Financial metrics (`Sharpe Ratio`, `Sortino Ratio`, `Max Drawdown`, `Directional Accuracy`)
    - Overfitting metrics (`R² gap`, `RMSE ratio`)
    - Overfitting status and model generalization label

- **Overfitting DataFrame:** An extendable CSV (`AllModel_OverfittingAnalysis.csv`) capturing overfitting analysis metrics across all models and variations.

- **Best Model per Algorithm:** The serialized best-performing variant of each algorithm for ensemble consideration or deployment.

- **Model Comparison:** A summary notebook or script that loads `AllModel_OverallPerformance.csv` and generates publication-ready comparison visualizations.

Together, these artifacts provide a complete, reproducible record of the modeling process, facilitating model tracking, comparison, selection, and deployment.

## Import Libraries and Root Configuration

In [1]:
""" Configure the utilities module path for imports """
import sys
import os
from pathlib import Path

# get project root as parent of current working directory
PROJECT_ROOT = Path(os.getcwd()).parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
# artifacts root
DATA_ROOT = PROJECT_ROOT / "data"
FEATURE_ROOT = PROJECT_ROOT / "artifacts" / "FeatureSelection"
FIGURE_ROOT = PROJECT_ROOT / "visualizations" / "ModelEvaluation"
MODEL_ROOT = PROJECT_ROOT / "artifacts" / "Models"
PERFORMANCE_ROOT = PROJECT_ROOT / "artifacts" / "ModelPerformance"

In [3]:
# records and calculations
import pandas as pd
import numpy as np

# avoid minor warnings
import warnings
warnings.filterwarnings('ignore')

# visualizations
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.base import clone
from scipy.stats import uniform, randint

# gradient boosting model
import xgboost as xgb

# optimization
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# import helper functions
from src.utilities import Evaluator, DataHandler, ModelPersister

## Load Dataset and Artifacts

In [4]:
df, x, y = DataHandler.load_dataset(DATA_ROOT / "gold_price_engineered.csv", target_col="target")
artifacts = DataHandler.load_artifacts(FEATURE_ROOT, cv_method='tscv')

In [5]:
# check dataset loading
df.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5,rolling_mean,rolling_std,momentum,target
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,0.013624,-0.018352,0.003223,-0.024552,...,0.001902,-0.002650,0.023711,-0.004229,-0.005142,0.008367,0.006266,0.014169,0.011275,0.003739
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.007948,0.013624,-0.018352,0.003223,...,0.000679,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.008043,0.012879,0.008881,0.010838
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,-0.013595,0.007948,0.013624,-0.018352,...,-0.004875,0.003739,0.019642,-0.002650,0.023711,-0.004229,0.011056,0.010901,0.015066,-0.017311
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010871,-0.013595,0.007948,0.013624,...,0.060478,0.010838,0.003739,0.019642,-0.002650,0.023711,0.002852,0.013993,-0.041022,-0.014661
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.024925,0.010871,-0.013595,0.007948,...,-0.058245,-0.017311,0.010838,0.003739,0.019642,-0.002650,0.000449,0.016053,-0.012010,-0.002307


In [6]:
# check input features
x.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag4,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5,rolling_mean,rolling_std,momentum
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,0.013624,-0.018352,0.003223,-0.024552,...,0.000679,0.001902,-0.002650,0.023711,-0.004229,-0.005142,0.008367,0.006266,0.014169,0.011275
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.007948,0.013624,-0.018352,0.003223,...,-0.004875,0.000679,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.008043,0.012879,0.008881
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,-0.013595,0.007948,0.013624,-0.018352,...,0.060478,-0.004875,0.003739,0.019642,-0.002650,0.023711,-0.004229,0.011056,0.010901,0.015066
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010871,-0.013595,0.007948,0.013624,...,-0.058245,0.060478,0.010838,0.003739,0.019642,-0.002650,0.023711,0.002852,0.013993,-0.041022
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.024925,0.010871,-0.013595,0.007948,...,0.009339,-0.058245,-0.017311,0.010838,0.003739,0.019642,-0.002650,0.000449,0.016053,-0.012010


In [7]:
# check target feature
y.head()

0    0.003739
1    0.010838
2   -0.017311
3   -0.014661
4   -0.002307
Name: target, dtype: float64

In [8]:
# load train/test split data
x_train, x_test, y_train, y_test = artifacts['x_train'], artifacts['x_test'], artifacts['y_train'], artifacts['y_test']
cv = artifacts['cv']

# **Baseline Modeling**

In [9]:
# model config
BASELINE_PARAMS = {
    'objective': 'reg:squarederror',
    'n_estimators': 100,
    'learning_rate': 0.1,
    'max_depth' : 6,
    'random_state': 42,
    'verbosity': 0
}

# train baseline model
baseline_model = xgb.XGBRegressor(**BASELINE_PARAMS)
baseline_model.fit(x_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


## Apply Model to Make Predication

In [10]:
# baseline prediction on train and test set
y_train_pred_baseline = baseline_model.predict(x_train)
y_test_pred_baseline = baseline_model.predict(x_test)

## Evaluating Model Performance
Calculating Errors for train and test data

In [11]:
# evaluation of train metrics
train_metrics_baseline = Evaluator.calculate_metrics(y_train, y_train_pred_baseline)

# evaluation of test metrics
test_metrics_baseline = Evaluator.calculate_metrics(y_test, y_test_pred_baseline)

In [12]:
# calculate directional time-series specific accuracy
train_acc_baseline = Evaluator.directional_accuracy(y_train, y_train_pred_baseline)
test_acc_baseline = Evaluator.directional_accuracy(y_test, y_test_pred_baseline)

In [13]:
# calcluate financial metrics of test data for baseline model
fin_baseline = Evaluator.financial_metrics('XGBoost (Baseline)', y_test, y_test_pred_baseline)

display(fin_baseline)

,Model,Sharpe Ratio,Sortino Ratio,Max Drawdown,Total Return (%)
0,XGBoost (Baseline),-1.8415,-2.6413,-37.767,-33.6179


In [14]:
# baseline model performance
baseline_perf = Evaluator.performance_table(train_metrics_baseline + [train_acc_baseline], test_metrics_baseline + [test_acc_baseline])

print("XGBoost - Baseline Modeling Performance")
display(baseline_perf)

XGBoost - Baseline Modeling Performance


,Metrics,Training,Test
0,MSE,0.0000,0.0001
1,MAE,0.0030,0.0066
2,RMSE,0.0041,0.0088
3,R2 Score,0.9108,-0.1872
4,MAPE,18888.7603,137432.2093
5,Directional Accuracy (%),92.1687,47.9212


# **Optuna Hyperparameter Optimzation**

## **Find Best Parameters**

In [15]:
# define objective function
def objective(trial):
    params = {
        "objective": "reg:squarederror",
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 10, log=True),
        "random_state": 42,
        "verbosity": 0,
        "n_jobs": -1
    }

    # crate model
    model = xgb.XGBRegressor(**params)

    # cross validation with TimeSeriesSplit
    cv_scores = []
    for train_idx, val_idx in cv.split(x_train, y_train):
        x_tr, x_val = x_train.iloc[train_idx], x_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        model_clone = clone(model)
        model_clone.fit(x_tr, y_tr)
        y_pred = model_clone.predict(x_val)
        
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        cv_scores.append(rmse)

    return np.mean(cv_scores)

In [16]:
# create study object
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))

# optimize hyperparameters
study.optimize(objective, n_trials=50, show_progress_bar=True)

# best parameter from optuna
best_params_optuna = study.best_params

Best trial: 47. Best value: 0.0117485: 100%|██████████| 50/50 [02:44<00:00,  3.29s/it]


## Train optimized model and Apply Model to Make Predictions

In [17]:
# train optimized model
optuna_model = xgb.XGBRegressor(**best_params_optuna, random_state=42, verbosity=0)

# fit optuna model
optuna_model.fit(x_train, y_train,verbose=False)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8613098408367447
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [18]:
# apply model to make predictions on train and test set
y_train_pred_optuna = optuna_model.predict(x_train)
y_test_pred_optuna = optuna_model.predict(x_test)

## Evaluating Optimized Model Performance
Calculating Errors for train and test data

In [19]:
""" calculate train and test metrics
return [mse, mae, rmse, r2, mape] """
train_metrics_optuna = Evaluator.calculate_metrics(y_train, y_train_pred_optuna)
test_metrics_optuna = Evaluator.calculate_metrics(y_test, y_test_pred_optuna)

In [20]:
# calculate directional time-series specific accuracy
train_acc_optuna = Evaluator.directional_accuracy(y_train, y_train_pred_optuna)
test_acc_optuna = Evaluator.directional_accuracy(y_test, y_test_pred_optuna)

In [21]:
# calcluate financial metrics of test data for Optuna model
fin_optuna = Evaluator.financial_metrics('XGBoost (Optuna)', y_test, y_test_pred_optuna)

display(fin_optuna)

,Model,Sharpe Ratio,Sortino Ratio,Max Drawdown,Total Return (%)
0,XGBoost (Optuna),0.101,0.1642,-17.6019,4.5879


In [22]:
# performance table of optuna optimization
optuna_perf = Evaluator.performance_table(train_metrics_optuna + [train_acc_optuna], test_metrics_optuna + [test_acc_optuna])

print("XGBoost - optuna Modeling Performance")
display(optuna_perf)

XGBoost - optuna Modeling Performance


,Metrics,Training,Test
0,MSE,0.0002,0.0001
1,MAE,0.0095,0.0060
2,RMSE,0.0139,0.0081
3,R2 Score,-0.0000,-0.0001
4,MAPE,4614.0165,9118.2591
5,Directional Accuracy (%),52.1906,51.2035


#  RandomizedSearchCV Optimzation

## Train RandomizedSearchCV Model

In [ ]:
# parameter distribution
RANDOM_PARAMS = {
    "n_estimators": randint(100, 1000),
    "learning_rate": uniform(0.01, 0.30),
    "max_depth": randint(3, 13),
    "min_child_weight": uniform(1, 9),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "gamma": uniform(0, 5),
    "reg_alpha": uniform(1e-6, 10),
    "reg_lambda": uniform(1e-6, 10),
}

# create empty model
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    verbosity=0
)

In [ ]:
# optimize with RandomSearchCV
random_search = RandomizedSearchCV(model,
                                   RANDOM_PARAMS,
                                   n_iter=50,
                                   scoring='neg_root_mean_squared_error',
                                   cv=cv,
                                   random_state=42,
                                   n_jobs=-1,
                                   verbose=0)

# fit model
random_search.fit(x_train, y_train)

# best model from RandomSearchCV
randomSearch_model = random_search.best_estimator_

## Apply Model to Make Predictions

In [ ]:
# make predictions on train and test set
y_train_pred_random = randomSearch_model.predict(x_train)
y_test_pred_random = randomSearch_model.predict(x_test)

## Evaluating Optimized Model Performance
Calculating Errors for train and test data

In [ ]:
# calculate train and test metrics
train_metrics_random = Evaluator.calculate_metrics(y_train, y_train_pred_random)
test_metrics_random = Evaluator.calculate_metrics(y_test, y_test_pred_random)

In [ ]:
# calculate directional time-series specific accuracy
train_acc_random = Evaluator.directional_accuracy(y_train, y_train_pred_random)
test_acc_random = Evaluator.directional_accuracy(y_test, y_test_pred_random)

In [ ]:
# calcluate financial metrics of test data for RandomSearch model
fin_random = Evaluator.financial_metrics('XGBoost (RandomSearchCV)', y_test, y_test_pred_random)

display(fin_random)

In [ ]:
# performance table of randomsearchCV optimization
random_perf = Evaluator.performance_table(train_metrics_random + [train_acc_random], test_metrics_random + [test_acc_random])

print("XGBoost - RandomSearchCV Optimized Modeling Performance")
display(random_perf)

# Cross Valiation (All Models)

In [ ]:
# cross validation of all models
cv_baseline = Evaluator.cv_evaluate(baseline_model, x_train, y_train, cv)
cv_optuna = Evaluator.cv_evaluate(optuna_model, x_train, y_train, cv)
cv_random = Evaluator.cv_evaluate(randomSearch_model, x_train, y_train, cv)

In [ ]:
# Evaluate cross-validation results
cv_df = pd.DataFrame({
    "Model": ["XGBoost (Baseline)", "XGBoost (Optuna)", "XGBoost (RandomSearchCV)"],
    "CV MSE": [cv_baseline['CV MSE'], cv_optuna['CV MSE'], cv_random['CV MSE']],
    "CV MAE": [cv_baseline['CV MAE'], cv_optuna['CV MAE'], cv_random['CV MAE']],
    "CV RMSE": [cv_baseline['CV RMSE'], cv_optuna['CV RMSE'], cv_random['CV RMSE']],
    "CV R2": [cv_baseline['CV R2'], cv_optuna['CV R2'], cv_random['CV R2']],
    "CV MAPE": [cv_baseline['CV MAPE'], cv_optuna['CV MAPE'], cv_random['CV MAPE']],
    "CV Directional Accuracy (%)": [cv_baseline['CV Directional Accuracy (%)'], cv_optuna['CV Directional Accuracy (%)'], cv_random['CV Directional Accuracy (%)']]
}).round(4)

print("Cross-Validation Results:")
display(cv_df)

# Summarize Models Performance

In [ ]:
# selected models
models = ['XGBoost (Baseline)', 'XGBoost (Optuna)', 'XGBoost (RandomSearchCV)']

# test metrics of all models
test_metrics = [test_metrics_baseline, test_metrics_optuna, test_metrics_random]

# test directional accuracy of all models
test_dir_acc = [test_acc_baseline, test_acc_optuna, test_acc_random]

# performance summary including cross validation results
performance_summary = Evaluator.summary_builder(models, cv_df, test_metrics, test_dir_acc)

In [ ]:
# Display the final performance summary table
print("Overall Model Performance:")
display(performance_summary)

# Overfitting Analysis

In [ ]:
# Overfitting analysis of all models

rows = []
idx_count = 0
for _, row in performance_summary.iterrows():
    # Extract metrics needed for assess_overfitting
    cv_r2 = row['CV R2']
    test_r2 = row['Test R2']
    cv_rmse = row['CV RMSE']
    test_rmse = row['Test RMSE']
    
    # get overfitting and generalization status
    gap, rmse_ratio, overfit_status, gen_status = Evaluator.assess_overfitting(
        cv_r2=cv_r2,
        test_r2=test_r2,
        cv_rmse=cv_rmse,
        test_rmse=test_rmse,
        tolerance=0.05
    )

    # directional accuracy (supplementary)
    cv_dir_acc = row.get('CV Directional Accuracy (%)', 0) 
    test_dir_acc = [test_acc_baseline, test_acc_optuna, test_acc_random][idx_count]
    dir_acc_gap = test_dir_acc - cv_dir_acc
    
    # build row
    rows.append({
        "Model": row['Model'],
        "CV RMSE": row['CV RMSE'],
        "CV R2": row['CV R2'],
        "CV Dir Acc (%)": cv_dir_acc,
        "Test RMSE": row['Test RMSE'],
        "Test R2": row['Test R2'],
        "Test Dir Acc (%)": test_dir_acc,
        "R2 Gap": gap,
        "RMSE Ratio": rmse_ratio,
        "Dir Acc Gap (%)": dir_acc_gap,
        "Overfitting Status": overfit_status,
        "Model Status (Generalization)": gen_status
    })

    idx_count += 1

overfit_df = pd.DataFrame(rows).round(4)

In [ ]:
print("XGBoost - Overfitting Analysis:")
display(overfit_df)

# **Visualizations of Model Comparison**

## **Standard Metrics Comparison**

In [ ]:
# figure configuration
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# model labels and colors
model_labels = ["Baseline", "Optuna", "RandomSearch"]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# metrics configuration
metrics = {
    "Test R2": {"title": "Test R² (↑ better)", "direction": "higher", "fmt": ".3f"},
    "Test RMSE": {"title": "Test RMSE (↓ better)", "direction": "lower", "fmt": ".4f"},
    "Test MAPE": {"title": "Test MAPE % (↓ better)", "direction": "lower", "fmt": ".2f"},
    "Test Dir Acc %": {"title": "Directional Accuracy % (↑ better)", "direction": "higher", "fmt": ".1f"}
}

# prepare values dictionary
values_dict = {
    "Test R2": performance_summary["Test R2"].values,
    "Test RMSE": performance_summary["Test RMSE"].values,
    "Test MAPE": performance_summary["Test MAPE"].values,
    "Test Dir Acc %": [test_acc_baseline, test_acc_optuna, test_acc_random]
}

for i, (metric, config) in enumerate(metrics.items()):
    ax = axes[i]
    values = values_dict[metric]
    
    # Create bar chart
    bars = ax.bar(model_labels, values, color=colors, edgecolor='black', linewidth=1.2)
    
    # add value labels on top of bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        label_pos = height + (max(values) * 0.02) if config["direction"] == "higher" else height + (max(values) * 0.01)
        ax.text(bar.get_x() + bar.get_width()/2., label_pos, f'{value:{config["fmt"]}}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # formatting
    ax.set_title(config["title"], fontsize=12, fontweight='bold')
    ax.set_ylabel('Value', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.set_axisbelow(True)
    
    # add horizontal reference line for "good" threshold
    if metric == "Test R2":
        ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Moderate threshold')
        ax.legend(fontsize=8, frameon=True)


plt.suptitle("XGBoost Model Comparison - Standard Metrics", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

# save figure
figure_path = FIGURE_ROOT / "01XGBoost_Comparison_StandardMetrics.png"
plt.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()

## **Financial Metrics Radar Chart**

In [ ]:
fin_metrics = ['Sharpe Ratio', 'Sortino Ratio', 'Max Drawdown', 'Total Return (%)']

# extract values
fin_values = {
    "Baseline": [
        fin_baseline['Sharpe Ratio'].values[0],
        fin_baseline['Sortino Ratio'].values[0],
        abs(fin_baseline['Max Drawdown'].values[0]),  # convert to positive for visualization
        fin_baseline['Total Return (%)'].values[0]
    ],
    "Optuna": [
        fin_optuna['Sharpe Ratio'].values[0],
        fin_optuna['Sortino Ratio'].values[0],
        abs(fin_optuna['Max Drawdown'].values[0]),
        fin_optuna['Total Return (%)'].values[0]
    ],
    "RandomSearch": [
        fin_random['Sharpe Ratio'].values[0],
        fin_random['Sortino Ratio'].values[0],
        abs(fin_random['Max Drawdown'].values[0]),
        fin_random['Total Return (%)'].values[0]
    ]
}

In [ ]:
# normalize metrics for radar chart
def normalize_metrics(values_dict, metrics, higher_is_better=None):
    
    if higher_is_better is None:
        # default: Sharpe/Sortino/Return = higher better, Drawdown = lower better
        higher_is_better = [True, True, False, True]
    
    normalized = {}
    
    for idx, metric in enumerate(metrics):
        all_values = [values_dict[model][idx] for model in values_dict]
        min_val, max_val = min(all_values), max(all_values)
        range_val = max_val - min_val if max_val != min_val else 1
        
        for model in values_dict:
            
            if model not in normalized:
                normalized[model] = []
            
            val = values_dict[model][idx]
            
            # normalize
            if higher_is_better[idx]:
                norm_val = (val - min_val) / range_val
            
            else:
                norm_val = 1 - (val - min_val) / range_val  # invert for "lower is better"
            
            normalized[model].append(norm_val)
    
    return normalized

In [ ]:
normalized_financial = normalize_metrics(fin_values, fin_metrics)

# create radar chart
fig, ax = plt.subplots(1, 1, figsize=(8, 8), subplot_kw=dict(projection='polar'))

# angle for each metric
angles = np.linspace(0, 2 * np.pi, len(fin_metrics), endpoint=False).tolist()
angles += angles[:1]                        # close the loop

# plot each model
for model, color in zip(model_labels, colors):
    values = normalized_financial[model]
    values += values[:1]                    # close the loop
    
    ax.plot(angles, values, 'o-', linewidth=2, label=model, color=color, markersize=6)
    ax.fill(angles, values, alpha=0.15, color=color)

# formatting
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(fin_metrics, size=11, weight='bold')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.title("Financial Metrics Comparison (Normalized)\nHigher is Better for All", fontsize=14, fontweight='bold', y=1.08)
plt.tight_layout()

# save figure
figure_path = FIGURE_ROOT / "01XGBoost_Comparison_FinancialMetrics.png"
plt.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()

# **Final Summary of Model Performances**

In [ ]:
# final summary of all models performance
final_summary = pd.merge(performance_summary,
                    overfit_df[['Model', 'R2 Gap', 'RMSE Ratio', "Dir Acc Gap (%)", 'Overfitting Status', 'Model Status (Generalization)']],
                    on="Model",
                    how="outer")

In [ ]:
print("Summary of The Models Performance:")
display(final_summary)

In [ ]:
# financial metrics of all variations
financial_metrics = pd.concat([fin_baseline, fin_optuna, fin_random], axis = 0).reset_index(drop=True)
financial_metrics

In [ ]:
# best model selection based on test R2 and test RMSE with tie-breaker logic
xgb_performance = final_summary.sort_values(
    by=['Test R2', 'Test RMSE'],
    ascending=[False, True]
)

best_variant = xgb_performance.iloc[0]

In [ ]:
# convert best variant performance to dataframe
best_variant = best_variant.to_frame(name='Value').rename_axis('Metrics').reset_index()

In [ ]:
best_variant

# **Persist Best Model and Performance**

In [ ]:
# model persister object
persister = ModelPersister(model_name = "XGBoost", model_root=MODEL_ROOT, performance_root=PERFORMANCE_ROOT)

In [ ]:
# persist performance of all variants
persister.save_performance(xgb_performance)

In [ ]:
# persists performance of the best variants from XGBoost
persister.save_performance(best_variant, "BestVariation")

In [ ]:
# persist financial metrics of XGBoost Model
persister.save_performance(financial_metrics, "FinancialMetrics")

In [ ]:
# aggregated model performance
persister.aggregated_performance(xgb_performance, "AllModel_OverallPerformance")

In [ ]:
# aggregated model performance
persister.aggregated_performance(financial_metrics, "AllModel_FinancialMetrics")

In [ ]:
# aggregated model performance
persister.aggregated_performance(xgb_performance.iloc[[0]], "AllModel_BestVariant")

In [ ]:
# save overfitting overfitting analysis
persister.append_overfitting(overfit_df)

In [ ]:
# save the best variant of XGBoost model
best_model_name = xgb_performance.iloc[0]['Model']

if "Optuna" in best_model_name:
    best_model = optuna_model

elif "RandomSearch" in best_model_name:
    best_model = randomSearch_model

else:
    best_model = baseline_model

persister.save_model(best_model)